[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab03.ipynb)

# Bab 3 — Citra Digital, Praproses, dan CNN: dari Nol vs *Transfer Learning*
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4** (*Runtime → Change runtime type → T4 GPU*), lalu **Runtime → Run all**.

Isi:
1. Citra sebagai larik: kanal, histogram, statistik normalisasi (BloodMNIST 64 px, 8 kelas sel darah).
2. Praproses: CLAHE dan ekualisasi histogram pada X-ray (PneumoniaMNIST), koreksi warna *gray-world* pada citra mikroskopi.
3. Augmentasi yang masuk akal secara klinis (flip/rotasi 90° boleh untuk sel darah; flip kiri–kanan **tidak** dianjurkan untuk X-ray dada).
4. Pelatihan dan perbandingan: CNN kecil dari nol, ResNet-18 bobot acak, ResNet-18 pra-latih ImageNet (dua tahap: beku lalu *fine-tune*),
   serta CNN tanpa augmentasi & tanpa *early stopping* untuk memperlihatkan *overfitting*.
5. *Learning curve*, *early stopping* (patience 5, maks. 30 epoch), matriks kebingungan, recall per kelas, efisiensi data (10 % data latih).

Semua hasil → `/content/hasil_bab03/` → `/content/hasil_bab03.zip`.

> Data: MedMNIST v2 (CC BY 4.0) — Yang dkk. (2023), *Scientific Data* 10, 41, https://doi.org/10.1038/s41597-022-01721-8.
> BloodMNIST berasal dari Acevedo dkk. (2020), *Data in Brief* 30, 105474. Bobot ImageNet ResNet-18 diunduh otomatis oleh `torchvision` dari `download.pytorch.org`.
> Model hanya untuk pembelajaran, **bukan** untuk penggunaan klinis.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 3.0 Lingkungan dan pencatat waktu

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB03_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB03_OUT", "/content/hasil_bab03"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
with catat_waktu("1_pip_install"):
    jalankan('pip install -q "medmnist==3.0.2"')
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib, matplotlib.ticker; import matplotlib.pyplot as plt
import torchvision, skimage, sklearn, medmnist
from skimage import exposure
from PIL import Image
from medmnist import INFO
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, recall_score
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = DEV.type == "cuda"
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
         "numpy": np.__version__, "scikit-image": skimage.__version__, "scikit-learn": sklearn.__version__,
         "medmnist": medmnist.__version__, "cuda": torch.version.cuda,
         "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))

SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()

## 3.1 Data dan citra sebagai larik

In [ ]:
UK = 28 if SMOKE else 64            # buku: 64 px; uji cepat penulis memakai 28 px
NAMA = ["basofil", "eosinofil", "eritroblas", "granulosit imatur", "limfosit", "monosit", "neutrofil", "trombosit"]
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn]); return np.load(folder / fn)
with catat_waktu("2_muat_data"):
    assert INFO["bloodmnist"]["license"] == "CC BY 4.0"
    b = npz_medmnist("bloodmnist", UK)
    D = {s: (b[f"{s}_images"], b[f"{s}_labels"].reshape(-1).astype(np.int64)) for s in ("train", "val", "test")}
    if SMOKE:
        r = np.random.default_rng(0)
        D = {s: (x[i], y[i]) for s, (x, y) in D.items() for i in [r.choice(len(y), {"train": 400, "val": 160, "test": 160}[s], replace=False)]}
    pn = npz_medmnist("pneumoniamnist", 128)
T31 = pd.DataFrame({s: np.bincount(D[s][1], minlength=8) for s in D}, index=NAMA); T31.loc["total"] = T31.sum()
T31.to_csv(OUT / "tabel_3_1_distribusi_kelas.csv"); print({s: D[s][0].shape for s in D}, D["train"][0].dtype); T31

In [ ]:
xtr = D["train"][0]
st = pd.DataFrame({"kanal": ["R", "G", "B"], "rerata_0_255": xtr.reshape(-1, 3).mean(0), "simpangan_baku_0_255": xtr.reshape(-1, 3).std(0)})
st["rerata_0_1"] = st.rerata_0_255 / 255; st["sb_0_1"] = st.simpangan_baku_0_255 / 255
st.to_csv(OUT / "tabel_3_2_statistik_kanal.csv", index=False)
contoh = xtr[np.where(D["train"][1] == 6)[0][0]]            # satu neutrofil
fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
ax[0].imshow(contoh); ax[0].set_title(f"RGB {contoh.shape}, {contoh.dtype}")
for k, (nm, cm) in enumerate([("R", "Reds"), ("G", "Greens"), ("B", "Blues")]):
    ax[k + 1].imshow(contoh[..., k], cmap=cm, vmin=0, vmax=255); ax[k + 1].set_title(f"kanal {nm}: [{contoh[..., k].min()}, {contoh[..., k].max()}]")
for k, c in enumerate("rgb"): ax[4].hist(contoh[..., k].ravel(), bins=32, range=(0, 255), color=c, alpha=0.5)
ax[4].set_title("histogram per kanal"); [a.axis("off") for a in ax[:4]]
plt.tight_layout(); plt.savefig(OUT / "gambar_3_1_larik_citra.png", bbox_inches="tight"); plt.show()
fig, ax = plt.subplots(2, 8, figsize=(12, 3.3)); r = np.random.default_rng(SEED)
for c in range(8):
    for rr in range(2):
        i = r.choice(np.where(D["train"][1] == c)[0]); ax[rr, c].imshow(xtr[i]); ax[rr, c].axis("off")
    ax[0, c].set_title(NAMA[c], fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "gambar_3_2_galeri_kelas.png", bbox_inches="tight"); plt.show()
st

## 3.2 Praproses: CLAHE, ekualisasi histogram, *gray-world*

In [ ]:
xr = pn["test_images"][0].astype(np.float32) / 255
def gray_world(img):
    f = img.astype(np.float32); m = f.reshape(-1, 3).mean(0); return np.clip(f * (m.mean() / m), 0, 255).astype(np.uint8)
fig, ax = plt.subplots(2, 4, figsize=(11, 5.6))
for a, (judul, im) in zip(ax[0], [("X-ray asli", xr), ("ekualisasi histogram", exposure.equalize_hist(xr)),
                                   ("CLAHE clip 0,01", exposure.equalize_adapthist(xr, clip_limit=0.01)),
                                   ("CLAHE clip 0,03", exposure.equalize_adapthist(xr, clip_limit=0.03))]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(judul); a.axis("off")
idx = [np.where(D["train"][1] == c)[0][0] for c in (1, 4)]
for j, i in enumerate(idx):
    ax[1, 2 * j].imshow(xtr[i]); ax[1, 2 * j].set_title(f"{NAMA[D['train'][1][i]]}: asli")
    ax[1, 2 * j + 1].imshow(gray_world(xtr[i])); ax[1, 2 * j + 1].set_title("gray-world")
for a in ax[1]: a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "gambar_3_3_praproses.png", bbox_inches="tight"); plt.show()

## 3.3 Augmentasi yang masuk akal secara klinis
Sel darah pada apusan tidak punya arah "atas" baku, jadi flip dan rotasi 90° aman. Pada X-ray dada, flip kiri–kanan memindahkan
bayangan jantung ke sisi kanan (menyerupai dekstrokardia) sehingga **tidak dianjurkan**; rotasi kecil dan perubahan kecerahan ringan lebih aman.

In [ ]:
def augmentasi_gpu(x, gen=None):
    # x: (N,3,H,W) float di [0,1]. Flip H/V acak per sampel, rotasi 90°·k per batch, jitter kecerahan/kontras ±10 %.
    n = x.shape[0]
    fh = torch.rand(n, device=x.device) < 0.5; x = torch.where(fh[:, None, None, None], x.flip(3), x)
    fv = torch.rand(n, device=x.device) < 0.5; x = torch.where(fv[:, None, None, None], x.flip(2), x)
    x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), (2, 3))
    kb = 1 + (torch.rand(n, 1, 1, 1, device=x.device) - 0.5) * 0.2; kc = 1 + (torch.rand(n, 1, 1, 1, device=x.device) - 0.5) * 0.2
    m = x.mean((1, 2, 3), keepdim=True); x = ((x - m) * kc + m) * kb
    return x.clamp(0, 1)
torch.manual_seed(SEED)
x0 = torch.tensor(xtr[idx[0]]).permute(2, 0, 1)[None].float().div(255).repeat(7, 1, 1, 1)
xa = torch.cat([augmentasi_gpu(x0[k:k + 1]) for k in range(7)])   # per sampel agar rotasinya beragam
fig, ax = plt.subplots(2, 8, figsize=(13, 3.6))
ax[0, 0].imshow(xtr[idx[0]]); ax[0, 0].set_title("asli", fontsize=8)
for k in range(7): ax[0, k + 1].imshow(xa[k].permute(1, 2, 0).numpy()); ax[0, k + 1].set_title(f"augmentasi {k+1}", fontsize=8)
xr8 = pn["test_images"][0]
pilihan = [("asli", xr8, "aman"), ("rotasi +7°", np.array(Image.fromarray(xr8).rotate(7, resample=Image.BILINEAR)), "aman"),
           ("rotasi −7°", np.array(Image.fromarray(xr8).rotate(-7, resample=Image.BILINEAR)), "aman"),
           ("kecerahan +10 %", np.clip(xr8 * 1.1, 0, 255), "aman"), ("kontras −10 %", np.clip((xr8 - xr8.mean()) * 0.9 + xr8.mean(), 0, 255), "aman"),
           ("geser 6 px", np.roll(xr8, 6, axis=1), "aman"), ("flip kiri–kanan", xr8[:, ::-1], "TIDAK dianjurkan"),
           ("flip atas–bawah", xr8[::-1], "TIDAK dianjurkan")]
for k, (j, im, ket) in enumerate(pilihan):
    ax[1, k].imshow(im, cmap="gray", vmin=0, vmax=255); ax[1, k].set_title(f"{j}\n({ket})", fontsize=7, color="red" if "TIDAK" in ket else "black")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "gambar_3_4_augmentasi.png", bbox_inches="tight"); plt.show()

## 3.4 Model dan *training loop* dengan *early stopping*
- **CNN kecil**: 4 blok Conv-BN-ReLU-MaxPool (32-64-128-256), GAP, dropout; masukan 64 px.
- **ResNet-18** (`torchvision`): masukan diperbesar ke 128 px di GPU dan dinormalisasi dengan statistik ImageNet; kepala diganti 8 kelas.
- Optimasi Adam, *mixed precision* (AMP) di GPU, *early stopping* pada *validation loss* (patience 5, maks. 30 epoch), bobot terbaik dipulihkan.

In [ ]:
MEAN_IN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SD_IN = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
class CNNKecil(nn.Module):
    def __init__(self, k=8):
        super().__init__()
        def blok(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.fitur = nn.Sequential(blok(3, 32), blok(32, 64), blok(64, 128), blok(128, 256))
        self.kepala = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, k))
    def forward(self, x): return self.kepala(self.fitur((x - 0.5) / 0.5))
class ResNet18(nn.Module):
    def __init__(self, pretrain, k=8, uk=128):
        super().__init__(); self.uk = uk
        w = None
        if pretrain:
            try: w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1; self.net = torchvision.models.resnet18(weights=w)
            except Exception as e:
                if not SMOKE: raise
                print("SMOKE: bobot ImageNet tidak dapat diunduh, memakai bobot acak:", e); self.net = torchvision.models.resnet18(weights=None)
        else: self.net = torchvision.models.resnet18(weights=None)
        self.net.fc = nn.Linear(self.net.fc.in_features, k)
    def forward(self, x):
        x = F.interpolate(x, size=(self.uk, self.uk), mode="bilinear", align_corners=False)
        return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))

XT = {s: torch.tensor(D[s][0]).permute(0, 3, 1, 2).contiguous() for s in D}       # uint8 di CPU
YT = {s: torch.tensor(D[s][1]) for s in D}
if DEV.type == "cuda": XT = {s: v.to(DEV) for s, v in XT.items()}; YT = {s: v.to(DEV) for s, v in YT.items()}   # data kecil: simpan di GPU
@torch.no_grad()
def evaluasi(model, s, idx=None, bs=1024):
    model.eval(); X, Y = XT[s], YT[s]; n = len(Y); out = []
    for i in range(0, n, bs):
        x = X[i:i + bs].to(DEV).float().div(255)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float())
    lg = torch.cat(out); loss = F.cross_entropy(lg, Y.to(DEV)).item()
    return loss, torch.softmax(lg, 1).cpu().numpy()

def latih(nama, model, idx_latih=None, maks_epoch=30, patience=5, aug=True, es=True, lr=1e-3, bs=128, tahap=None, seed=SEED):
    seed_semua(seed); model = model.to(DEV)
    n_all = len(YT["train"]); idx_latih = np.arange(n_all) if idx_latih is None else idx_latih
    if SMOKE: maks_epoch, patience = min(maks_epoch, 3), 1
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); log = []; terbaik = (np.inf, None, -1); ep_global = 0
    for (n_ep, lr_t, beku) in (tahap or [(maks_epoch, lr, False)]):
        for nm, p in model.named_parameters(): p.requires_grad = (not beku) or nm.startswith("net.fc")
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr_t); tunggu = 0
        for ep in range(n_ep if not SMOKE else min(n_ep, 3)):
            model.train(); t0 = time.perf_counter(); perm = np.random.permutation(idx_latih); tl, tb, n = 0.0, 0, 0
            for i in range(0, len(perm), bs):
                b = torch.as_tensor(perm[i:i + bs], device=XT["train"].device)
                x = XT["train"][b].to(DEV).float().div(255); y = YT["train"][b].to(DEV)
                if aug: x = augmentasi_gpu(x)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): lg = model(x); loss = F.cross_entropy(lg, y)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(b); tb += (lg.argmax(1) == y).sum().item(); n += len(b)
            if DEV.type == "cuda": torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            vl, vp = evaluasi(model, "val"); va = accuracy_score(D["val"][1], vp.argmax(1)); ep_global += 1
            log.append({"model": nama, "epoch": ep_global, "tahap_beku": beku, "loss_latih": tl / n, "akurasi_latih": tb / n,
                        "loss_val": vl, "akurasi_val": va, "detik_epoch": dt})
            print(f"{nama} ep{ep_global:02d} loss {tl/n:.4f}/{vl:.4f} acc {tb/n:.4f}/{va:.4f} {dt:.1f}s")
            if vl < terbaik[0]:
                terbaik = (vl, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep_global); tunggu = 0
            else:
                tunggu += 1
                if es and not beku and tunggu >= patience: print(f"  early stopping: tidak membaik {patience} epoch"); break
    if es and terbaik[1] is not None: model.load_state_dict(terbaik[1])
    return model, pd.DataFrame(log), (terbaik[2] if es else ep_global)

def metrik_uji(nama, model, ep_terbaik, log, catatan=""):
    _, p = evaluasi(model, "test"); y = D["test"][1]; yh = p.argmax(1)
    try: auc = roc_auc_score(y, p, multi_class="ovr", average="macro")
    except ValueError: auc = float("nan")
    return {"model": nama, "parameter": sum(q.numel() for q in model.parameters()), "epoch_dijalankan": int(log.epoch.max()),
            "epoch_terbaik": int(ep_terbaik), "detik_per_epoch": float(log.detik_epoch.mean()), "detik_total": float(log.detik_epoch.sum()),
            "akurasi": accuracy_score(y, yh), "akurasi_seimbang": balanced_accuracy_score(y, yh), "F1_makro": f1_score(y, yh, average="macro"),
            "AUC_makro_OvR": auc, "catatan": catatan}, p

## 3.5 Eksperimen utama

In [ ]:
HASIL, LOG, PROB, MODEL = [], [], {}, {}
def jalankan_eksperimen(nama, buat_model, catatan, **kw):
    with catat_waktu(f"latih_{nama}"):
        m, lg, ep = latih(nama, buat_model(), **kw)
    r, p = metrik_uji(nama, m, ep, lg, catatan); HASIL.append(r); LOG.append(lg); PROB[nama] = p; MODEL[nama] = m
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
jalankan_eksperimen("CNN kecil (dari nol)", CNNKecil, "augmentasi + early stopping")
jalankan_eksperimen("CNN kecil tanpa aug & tanpa ES", CNNKecil, "30 epoch penuh, bobot epoch terakhir", aug=False, es=False)
jalankan_eksperimen("ResNet-18 bobot acak", lambda: ResNet18(False), "augmentasi + early stopping")
jalankan_eksperimen("ResNet-18 pra-latih ImageNet", lambda: ResNet18(True), "tahap 1: 3 epoch beku (hanya fc, lr 1e-3); tahap 2: fine-tune penuh lr 1e-4 + ES",
                    tahap=[(3, 1e-3, True), (30, 1e-4, False)])
T33 = pd.DataFrame(HASIL); T33.to_csv(OUT / "tabel_3_3_perbandingan_model.csv", index=False)
pd.concat(LOG).to_csv(OUT / "log_epoch_semua_model.csv", index=False)
torch.save(MODEL["CNN kecil (dari nol)"].state_dict(), OUT / "cnn_kecil_blood_terbaik.pt")
T33

## 3.6 *Learning curve* dan *early stopping*

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for a, lg in zip(ax.ravel(), LOG):
    nama = lg.model.iloc[0]; r = next(h for h in HASIL if h["model"] == nama)
    a.plot(lg.epoch, lg.loss_latih, "o-", ms=3, label="loss latih"); a.plot(lg.epoch, lg.loss_val, "s-", ms=3, label="loss validasi")
    a2 = a.twinx(); a2.plot(lg.epoch, lg.akurasi_val, "^-", ms=3, color="tab:green", alpha=0.7, label="akurasi validasi"); a2.set_ylim(0, 1)
    a2.set_ylabel("akurasi val", color="tab:green")
    if r["catatan"].startswith("30 epoch") is False: a.axvline(r["epoch_terbaik"], color="red", ls="--", lw=1, label=f"bobot terbaik (ep {r['epoch_terbaik']})")
    if lg.tahap_beku.any(): a.axvspan(0.5, lg[lg.tahap_beku].epoch.max() + 0.5, color="gray", alpha=0.15, label="tahap beku")
    a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True)); a.set_title(nama, fontsize=9); a.set_xlabel("epoch"); a.set_ylabel("cross-entropy"); a.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.savefig(OUT / "gambar_3_5_learning_curve.png", bbox_inches="tight"); plt.show()

## 3.7 Matriks kebingungan dan recall per kelas

In [ ]:
y = D["test"][1]; pilih = ["CNN kecil (dari nol)", "ResNet-18 pra-latih ImageNet"]
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, nm in zip(ax, pilih):
    cm = confusion_matrix(y, PROB[nm].argmax(1), labels=range(8)); cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(8):
        for j in range(8): a.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > 0.6 else "black")
    a.set_xticks(range(8)); a.set_yticks(range(8)); a.set_xticklabels(NAMA, rotation=45, ha="right", fontsize=7); a.set_yticklabels(NAMA, fontsize=7)
    a.set_xlabel("prediksi"); a.set_ylabel("sebenarnya"); a.set_title(nm, fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "gambar_3_6_matriks_kebingungan.png", bbox_inches="tight"); plt.show()
T34 = pd.DataFrame({nm: recall_score(y, PROB[nm].argmax(1), labels=range(8), average=None, zero_division=0) for nm in PROB}, index=NAMA)
T34.insert(0, "n_uji", np.bincount(y, minlength=8)); T34.to_csv(OUT / "tabel_3_4_recall_per_kelas.csv"); T34.round(4)

## 3.8 Efisiensi data: hanya 10 % data latih

In [ ]:
from sklearn.model_selection import train_test_split
idx10, _ = train_test_split(np.arange(len(D["train"][1])), train_size=0.10, stratify=D["train"][1], random_state=SEED)
HASIL_10 = []
for nama, bm, kw in [("CNN kecil (dari nol)", CNNKecil, {}),
                     ("ResNet-18 pra-latih ImageNet", lambda: ResNet18(True), {"tahap": [(3, 1e-3, True), (30, 1e-4, False)]})]:
    with catat_waktu(f"latih10_{nama}"):
        m, lg, ep = latih(nama + " [10%]", bm(), idx_latih=idx10, **kw)
    r, _ = metrik_uji(nama, m, ep, lg, "10 % data latih"); r["fraksi_latih"] = 0.10; r["n_latih"] = len(idx10); HASIL_10.append(r); LOG.append(lg)
    full = next(h for h in HASIL if h["model"] == nama); full = dict(full, fraksi_latih=1.0, n_latih=len(D["train"][1])); HASIL_10.append(full)
T35 = pd.DataFrame(HASIL_10)[["model", "fraksi_latih", "n_latih", "epoch_terbaik", "akurasi", "akurasi_seimbang", "F1_makro", "AUC_makro_OvR", "detik_total"]]
T35.to_csv(OUT / "tabel_3_5_efisiensi_data.csv", index=False)
pd.concat(LOG).to_csv(OUT / "log_epoch_semua_model.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 3.4))
for nm, mk in [("CNN kecil (dari nol)", "o-"), ("ResNet-18 pra-latih ImageNet", "s-")]:
    d = T35[T35.model == nm].sort_values("fraksi_latih"); ax.plot(d.n_latih, d.F1_makro, mk, label=nm)
    for _, rr in d.iterrows(): ax.annotate(f"{rr.F1_makro:.3f}", (rr.n_latih, rr.F1_makro), textcoords="offset points", xytext=(4, -10), fontsize=7)
ax.set_xscale("log"); ax.set_xlabel("jumlah citra latih (skala log)"); ax.set_ylabel("F1 makro (uji)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "gambar_3_7_efisiensi_data.png", bbox_inches="tight"); plt.show()
T35

## 3.9 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(ukuran_px=UK, n={s: int(len(D[s][1])) for s in D}, perbandingan=T33.to_dict("records"), efisiensi_data=T35.to_dict("records"), statistik_kanal=st.to_dict("records")))))
(OUT / "ringkasan_bab03.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass